# Bias Assessment Results Analyzer

This notebook analyzes the output from the bias assessment tool to calculate and compare bias detection rates between Human and DAX notes.

## Usage

1. Set `BIAS_NOTES_RESULTS_FILE` in your local `.env`
2. Run all cells to see the analysis


## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Set the path to your results file via .env
# =============================================================================

from project_config import BIAS_NOTES_RESULTS_FILE

RESULTS_FILE = BIAS_NOTES_RESULTS_FILE
if not RESULTS_FILE:
    raise RuntimeError(
        "Set BIAS_NOTES_RESULTS_FILE in .env before running this notebook."
    )

## Load and Analyze Results

In [ ]:
import pandas as pd
import json

# Load the results file
print(f"Loading results from: {RESULTS_FILE}")
df = pd.read_csv(RESULTS_FILE)
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")

In [ ]:
# =============================================================================
# RESULTS SUMMARY - Bias Detection Rates by Note Type (Human vs DAX)
# =============================================================================

print("=" * 70)
print("BIAS DETECTION RESULTS SUMMARY")
print("=" * 70)
print(f"\nResults file: {RESULTS_FILE}")


# Helper function to check if a JSON array string is non-empty
def has_terms(json_str):
    if pd.isna(json_str):
        return False
    return json_str != "[]" and json_str != ""


# Check which columns exist
has_possible_col = "Possible_Biased_Terms" in df.columns
has_likely_col = "Likely_Biased_Terms" in df.columns
has_legacy_col = "Biased_Terms" in df.columns  # For older output files

# Add helper columns for analysis
if has_possible_col:
    df["has_possible"] = df["Possible_Biased_Terms"].apply(has_terms)
else:
    df["has_possible"] = False

if has_likely_col:
    df["has_likely"] = df["Likely_Biased_Terms"].apply(has_terms)
elif has_legacy_col:
    # Use legacy column if new columns don't exist
    df["has_likely"] = df["Biased_Terms"].apply(has_terms)
    print("\n[INFO] Using legacy 'Biased_Terms' column (treating as 'likely' bias)")
else:
    df["has_likely"] = False

df["has_any_bias"] = df["has_possible"] | df["has_likely"]

# Overall statistics
total_notes = len(df)
notes_with_possible = df["has_possible"].sum()
notes_with_likely = df["has_likely"].sum()
notes_with_any = df["has_any_bias"].sum()

print(f"\n{'OVERALL RESULTS':^70}")
print("-" * 70)
print(f"Total notes analyzed: {total_notes}")
print("")
if has_possible_col:
    print(
        f"  Notes with POSSIBLE bias:  {notes_with_possible:4d}  ({100 * notes_with_possible / total_notes:5.1f}%)"
    )
print(
    f"  Notes with LIKELY bias:    {notes_with_likely:4d}  ({100 * notes_with_likely / total_notes:5.1f}%)"
)
print(
    f"  Notes with ANY bias:       {notes_with_any:4d}  ({100 * notes_with_any / total_notes:5.1f}%)"
)

# Check if Dax_or_Human column exists for breakdown
if "Dax_or_Human" in df.columns:
    print(f"\n{'RESULTS BY NOTE TYPE':^70}")
    print("-" * 70)

    # Get unique note types
    note_types = df["Dax_or_Human"].unique()

    # Create summary table
    summary_data = []

    for note_type in sorted(note_types):
        subset = df[df["Dax_or_Human"] == note_type]
        n = len(subset)
        n_possible = subset["has_possible"].sum()
        n_likely = subset["has_likely"].sum()
        n_any = subset["has_any_bias"].sum()

        row_data = {
            "Note Type": note_type,
            "Total": n,
        }
        if has_possible_col:
            row_data["Possible Bias"] = n_possible
            row_data["Possible %"] = f"{100 * n_possible / n:.1f}%" if n > 0 else "N/A"
        row_data["Likely Bias"] = n_likely
        row_data["Likely %"] = f"{100 * n_likely / n:.1f}%" if n > 0 else "N/A"
        row_data["Any Bias"] = n_any
        row_data["Any %"] = f"{100 * n_any / n:.1f}%" if n > 0 else "N/A"

        summary_data.append(row_data)

        print(f"\n  {note_type.upper()} Notes (n={n}):")
        if has_possible_col:
            print(
                f"    Possible bias: {n_possible:4d}  ({100 * n_possible / n:5.1f}%)"
                if n > 0
                else "    Possible bias: N/A"
            )
        print(
            f"    Likely bias:   {n_likely:4d}  ({100 * n_likely / n:5.1f}%)"
            if n > 0
            else "    Likely bias:   N/A"
        )
        print(
            f"    Any bias:      {n_any:4d}  ({100 * n_any / n:5.1f}%)"
            if n > 0
            else "    Any bias:      N/A"
        )

    # Display as DataFrame table
    print(f"\n{'COMPARISON TABLE':^70}")
    print("-" * 70)
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)

    # Calculate and display difference if both Human and Dax exist
    if "Human" in note_types and "Dax" in note_types:
        human_df = df[df["Dax_or_Human"] == "Human"]
        dax_df = df[df["Dax_or_Human"] == "Dax"]

        human_any_rate = (
            100 * human_df["has_any_bias"].sum() / len(human_df)
            if len(human_df) > 0
            else 0
        )
        dax_any_rate = (
            100 * dax_df["has_any_bias"].sum() / len(dax_df) if len(dax_df) > 0 else 0
        )

        human_likely_rate = (
            100 * human_df["has_likely"].sum() / len(human_df)
            if len(human_df) > 0
            else 0
        )
        dax_likely_rate = (
            100 * dax_df["has_likely"].sum() / len(dax_df) if len(dax_df) > 0 else 0
        )

        if has_possible_col:
            human_possible_rate = (
                100 * human_df["has_possible"].sum() / len(human_df)
                if len(human_df) > 0
                else 0
            )
            dax_possible_rate = (
                100 * dax_df["has_possible"].sum() / len(dax_df)
                if len(dax_df) > 0
                else 0
            )

        print(f"\n{'HUMAN vs DAX COMPARISON':^70}")
        print("-" * 70)
        print("\n  Bias Rate Comparison (Human - DAX):")
        print("")
        if has_possible_col:
            print(
                f"    Possible bias: {human_possible_rate:5.1f}% vs {dax_possible_rate:5.1f}%  (diff: {human_possible_rate - dax_possible_rate:+.1f} pp)"
            )
        print(
            f"    Likely bias:   {human_likely_rate:5.1f}% vs {dax_likely_rate:5.1f}%  (diff: {human_likely_rate - dax_likely_rate:+.1f} pp)"
        )
        print(
            f"    Any bias:      {human_any_rate:5.1f}% vs {dax_any_rate:5.1f}%  (diff: {human_any_rate - dax_any_rate:+.1f} pp)"
        )
        print("\n  (pp = percentage points)")

else:
    print("\n[INFO] Column 'Dax_or_Human' not found - skipping breakdown by note type.")
    print(
        "       To see Human vs DAX comparison, ensure your input CSV has a 'Dax_or_Human' column."
    )

## Sample Flagged Notes

In [ ]:
print(f"{'SAMPLE FLAGGED NOTES':^70}")
print("-" * 70)

# Show notes that have at least one flagged term
flagged = df[df["has_any_bias"]]
print(f"Notes with flagged terms: {len(flagged)} / {len(df)}")

if len(flagged) > 0:
    print("\nFirst 10 flagged notes:")
    display_cols = []
    if "Dax_or_Human" in df.columns:
        display_cols.append("Dax_or_Human")
    if "Likely_Biased_Terms" in df.columns:
        display_cols.append("Likely_Biased_Terms")
    if "Possible_Biased_Terms" in df.columns:
        display_cols.append("Possible_Biased_Terms")
    if "Biased_Terms" in df.columns and "Likely_Biased_Terms" not in df.columns:
        display_cols.append("Biased_Terms")

    display(flagged[display_cols].head(10))

## Most Common Biased Terms

In [ ]:
from collections import Counter


def extract_terms(series):
    """Extract all terms from a series of JSON array strings."""
    all_terms = []
    for val in series:
        if pd.isna(val) or val == "[]" or val == "":
            continue
        try:
            terms = json.loads(val)
            if isinstance(terms, list):
                all_terms.extend(terms)
        except json.JSONDecodeError:
            pass
    return all_terms


print(f"{'MOST COMMON BIASED TERMS':^70}")
print("-" * 70)

# Analyze likely bias terms
likely_col = (
    "Likely_Biased_Terms" if "Likely_Biased_Terms" in df.columns else "Biased_Terms"
)
if likely_col in df.columns:
    likely_terms = extract_terms(df[likely_col])
    likely_counts = Counter(likely_terms)

    print(f"\nTop 15 LIKELY bias terms (total occurrences: {len(likely_terms)}):")
    for term, count in likely_counts.most_common(15):
        print(f"  {count:4d}x  {term}")

# Analyze possible bias terms
if "Possible_Biased_Terms" in df.columns:
    possible_terms = extract_terms(df["Possible_Biased_Terms"])
    possible_counts = Counter(possible_terms)

    print(f"\nTop 15 POSSIBLE bias terms (total occurrences: {len(possible_terms)}):")
    for term, count in possible_counts.most_common(15):
        print(f"  {count:4d}x  {term}")

# Breakdown by note type if available
if "Dax_or_Human" in df.columns:
    print(f"\n{'TERM FREQUENCY BY NOTE TYPE':^70}")
    print("-" * 70)

    for note_type in sorted(df["Dax_or_Human"].unique()):
        subset = df[df["Dax_or_Human"] == note_type]
        terms = extract_terms(
            subset[likely_col] if likely_col in subset.columns else pd.Series()
        )
        counts = Counter(terms)

        print(f"\n{note_type.upper()} - Top 10 likely bias terms:")
        for term, count in counts.most_common(10):
            print(f"  {count:4d}x  {term}")